# Extract Attention Matrices - Gemma 3 with Memory Validation

This notebook demonstrates how to:
1. Load Gemma 3 models with **memory validation** (prevents OOM)
2. Use **quantization** (int4/int8) for reduced memory
3. Analyze attention with **top N tokens only** (memory-efficient)
4. Filter out **chat template tokens** from analysis

In [ ]:
import os

import torch
from dotenv import load_dotenv
from huggingface_hub import login

from model_evaluation.main_agent import (
    GemmaModelConfig,
    MemoryTracker,
    analyze_prompt_lite,
    get_layer_types,
    load_gemma_model,
    print_memory_usage,
    print_model_info,
    print_model_memory_footprint,
    validate_memory_for_context,
)
from model_evaluation.main_agent.example_prompts import (
    get_markdown_prompts,
    get_plain_prompts,
    get_simple_prompts,
)

load_dotenv()
# but setting grad_enabled(False) is still a good safety measure.
torch.set_grad_enabled(False)

if os.getenv("HF_TOKEN"):
    login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
MODEL_ID = "google/gemma-3-4b-it"  # Start with 1B for testing
# MODEL_ID = "google/gemma-3-4b-it"  # Uncomment for 4B (needs ~17GB VRAM)

print_model_info(MODEL_ID, quantization="int4")

# Check if we have enough memory
is_safe, breakdown = validate_memory_for_context(MODEL_ID, context_length=8192, quantization="int4")
print(f"\nMemory Safe: {is_safe}")
print(f"Required: {breakdown['min_required_gb']:.1f} GB")
print(f"Available: {breakdown['available_gb']:.1f} GB")


════════════════════════════════════════════════════════════
📊 Gemma 3 4B Model Info
════════════════════════════════════════════════════════════

🏗️  Architecture:
    Parameters:     4.0B
    Layers:         34 (5 global, 29 local)
    Heads:          16 (KV: 8)
    Sliding Window: 1024

💾 Memory (int4, 32k context):
    Weights:        2.6 GB
    KV Cache:       1.5 GB
    Attn (global):  64.0 GB/layer
════════════════════════════════════════════════════════════


Memory Safe: True
Required: 7.1 GB
Available: 12.0 GB


In [5]:
config = GemmaModelConfig(
    model_id=MODEL_ID,
    quantization="int4",  # Set to "int4" for ~3x memory reduction
    max_context_length=8192,
)

print(f"\n{'=' * 60}")
print(f"Loading {config.model_id}")
print(f"Quantization: {config.quantization or 'None (bf16)'}")
print("attn_implementation: eager (required for attention extraction)")
print(f"{'=' * 60}")

with MemoryTracker(f"Loading {config.model_id}"):
    model, tokenizer = load_gemma_model(config)
    model.eval()

print_model_memory_footprint(model, config.model_id)

# Device for analysis
device = str(next(model.parameters()).device)
print(f"Model loaded on: {device}")


Loading google/gemma-3-4b-it
Quantization: int4
attn_implementation: eager (required for attention extraction)
📦 Using Unsloth pre-quantized model: unsloth/gemma-3-4b-it-bnb-4bit


`torch_dtype` is deprecated! Use `dtype` instead!



──────────────────────────────────────────────────
📊 Memory Tracking: Loading google/gemma-3-4b-it
──────────────────────────────────────────────────
  ⏱️  Duration:    1.80s
  💾 RSS Change:  +80.88 MB
  🖥️  CUDA Change: +0.00 MB
  📈 Peak CUDA:   0.00 MB
──────────────────────────────────────────────────



TypeError: 'NoneType' object is not subscriptable

In [ ]:
layer_types = get_layer_types(MODEL_ID)
num_global = sum(1 for t in layer_types.values() if t == "global")
num_local = sum(1 for t in layer_types.values() if t == "local")

print("\nLayer Distribution:")
print(f"  Global layers: {num_global} (quadratic attention)")
print(f"  Local layers:  {num_local} (sliding window)")
print(f"  Global indices: {[i for i, t in layer_types.items() if t == 'global']}")

In [ ]:
simple_prompts = get_simple_prompts(tokenizer)
markdown_prompts = get_markdown_prompts(tokenizer)
plain_prompts = get_plain_prompts(tokenizer)

print(f"Simple prompts: {list(simple_prompts.keys())}")
print(f"Markdown prompts: {list(markdown_prompts.keys())}")
print(f"Plain prompts: {list(plain_prompts.keys())}")

## Memory-Efficient Analysis with Top N Tokens

Using `analyze_prompt_lite()` which:
- Returns only **top N** most important tokens
- **Excludes chat template tokens** (start_of_turn, end_of_turn, etc.)
- Uses **O(N)** memory instead of O(seq²)

In [ ]:
for key, text in simple_prompts.items():
    result_df = analyze_prompt_lite(
        key=key,
        prompt=text,
        model=model,
        tokenizer=tokenizer,
        device=device,
        top_n=10,
        exclude_template=True,
    )

## Agentic RAG Analysis

Analyze the full agentic prompts with tool calls and retrieved documents.

In [ ]:
question_key = "deadline"
md_prompt = markdown_prompts[f"md_{question_key}"]

print(f"Markdown prompt tokens: {len(tokenizer.encode(md_prompt))}")

result_md = analyze_prompt_lite(
    key=f"md_{question_key}",
    prompt=md_prompt,
    model=model,
    tokenizer=tokenizer,
    device=device,
    top_n=20,
    exclude_template=True,
)

In [ ]:
plain_prompt = plain_prompts[f"plain_{question_key}"]

print(f"Plain text prompt tokens: {len(tokenizer.encode(plain_prompt))}")

result_plain = analyze_prompt_lite(
    key=f"plain_{question_key}",
    prompt=plain_prompt,
    model=model,
    tokenizer=tokenizer,
    device=device,
    top_n=20,
    exclude_template=True,
)

## Compare Markdown vs Plain Text

In [ ]:
print("\n" + "=" * 60)
print("Markdown vs Plain Text - Top Tokens Comparison")
print("=" * 60)

print("\nMarkdown top 5:")
print(result_md.head(5).to_string(index=False))

print("\nPlain text top 5:")
print(result_plain.head(5).to_string(index=False))

## Final Memory State

In [ ]:
print_memory_usage("Final State")